# Notebook 18 - Higher-Order Functions and JSON Parsing

**Datasets:** `samples.bakehouse.sales_transactions`, `samples.bakehouse.sales_customers`, `samples.wanderbricks.bookings`  
**Difficulty:** Medium  
**Topics:** `transform`, `filter` (higher-order), `aggregate`, `exists`, `forall`, `zip_with`, `array_distinct`, `array_sort`, `array_size`, `array_join`, `from_json`, `schema_of_json`, `create_map`, `map_keys`, `map_values`, `map_from_arrays`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_bookings     = spark.table("samples.wanderbricks.bookings")

## Problem 1 - transform

Create the following test DataFrame:

```python
df_prices = spark.createDataFrame([(1, [10, 20, 30]), (2, [5, 15, 25])], ['id', 'prices'])
```

Use `F.transform()` with a lambda to double each element in the `prices` array.  
Store the result in a new column called `doubled_prices`.  
**Expected columns:** `id`, `prices`, `doubled_prices`

In [ ]:
result_1 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 1 ---------
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'id' in cols, "Missing: id"
assert 'prices' in cols, "Missing: prices"
assert 'doubled_prices' in cols, "Missing: doubled_prices"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
rows = result_1.orderBy('id').collect()
assert rows[0]['doubled_prices'][0] == 20, "First element of doubled_prices for id=1 should be 20"
assert rows[0]['doubled_prices'][1] == 40, "Second element of doubled_prices for id=1 should be 40"
assert rows[0]['doubled_prices'][2] == 60, "Third element of doubled_prices for id=1 should be 60"
print(f"Problem 1 passed  ({len(rows)} rows)")

## Problem 2 - filter (higher-order)

Using the same test DataFrame from Problem 1 (or recreate it):

```python
df_prices = spark.createDataFrame([(1, [10, 20, 30]), (2, [5, 15, 25])], ['id', 'prices'])
```

Use the higher-order `F.filter()` with a lambda to keep only elements greater than 12 from the `prices` array.  
Store the result in a new column called `filtered_prices`.  
**Expected columns:** `id`, `prices`, `filtered_prices`

In [ ]:
result_2 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 2 ---------
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'id' in cols, "Missing: id"
assert 'prices' in cols, "Missing: prices"
assert 'filtered_prices' in cols, "Missing: filtered_prices"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
rows = result_2.orderBy('id').collect()
for row in rows:
    for val in row['filtered_prices']:
        assert val > 12, f"filtered_prices contains value {val} which is not > 12"
# row with id=1 should have all 3 elements (10 is dropped, 20 and 30 kept -> wait, 10 < 12 so dropped)
assert rows[0]['filtered_prices'] == [20, 30], "id=1 filtered_prices should be [20, 30]"
print(f"Problem 2 passed  ({len(rows)} rows)")

## Problem 3 - aggregate (higher-order)

Create the following test DataFrame:

```python
df_vals = spark.createDataFrame([(1, [3, 5, 7]), (2, [10, 20])], ['id', 'vals'])
```

Use `F.aggregate()` with a zero initial value and an accumulator lambda to compute the sum of each array without using `explode`.  
Store the result in a new column called `array_sum`.  
**Expected columns:** `id`, `vals`, `array_sum`

In [ ]:
result_3 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 3 ---------
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'id' in cols, "Missing: id"
assert 'vals' in cols, "Missing: vals"
assert 'array_sum' in cols, "Missing: array_sum"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
rows = result_3.orderBy('id').collect()
assert rows[0]['array_sum'] == 15, f"id=1 array_sum should be 15, got {rows[0]['array_sum']}"
assert rows[1]['array_sum'] == 30, f"id=2 array_sum should be 30, got {rows[1]['array_sum']}"
print(f"Problem 3 passed  ({len(rows)} rows)")

## Problem 4 - exists and forall

Create the following test DataFrame:

```python
df_nums = spark.createDataFrame([(1, [2, 4, 6]), (2, [1, 4, 6]), (3, [1, 3, 5])], ['id', 'nums'])
```

Use `F.exists()` to add a boolean column `any_odd` indicating whether any element in `nums` is odd.  
Use `F.forall()` to add a boolean column `all_even` indicating whether all elements in `nums` are even.  
**Expected columns:** `id`, `nums`, `any_odd`, `all_even`

In [ ]:
result_4 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 4 ---------
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'id' in cols, "Missing: id"
assert 'nums' in cols, "Missing: nums"
assert 'any_odd' in cols, "Missing: any_odd"
assert 'all_even' in cols, "Missing: all_even"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
assert result_4.schema['any_odd'].dataType.typeName() == 'boolean', "any_odd must be BooleanType"
assert result_4.schema['all_even'].dataType.typeName() == 'boolean', "all_even must be BooleanType"
rows = result_4.orderBy('id').collect()
# id=1: [2,4,6] - no odd, all even
assert rows[0]['any_odd'] == False, "id=1 any_odd should be False"
assert rows[0]['all_even'] == True, "id=1 all_even should be True"
# id=2: [1,4,6] - has odd, not all even
assert rows[1]['any_odd'] == True, "id=2 any_odd should be True"
assert rows[1]['all_even'] == False, "id=2 all_even should be False"
# id=3: [1,3,5] - all odd, not all even
assert rows[2]['any_odd'] == True, "id=3 any_odd should be True"
assert rows[2]['all_even'] == False, "id=3 all_even should be False"
print(f"Problem 4 passed  ({len(rows)} rows)")

## Problem 5 - zip_with

Create the following test DataFrame:

```python
df_zip = spark.createDataFrame([(1, [1, 2, 3], [10, 20, 30])], ['id', 'a', 'b'])
```

Use `F.zip_with()` with a lambda that multiplies corresponding elements from arrays `a` and `b`.  
Store the result in a new column called `products`.  
**Expected columns:** `id`, `a`, `b`, `products`

In [ ]:
result_5 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 5 ---------
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'id' in cols, "Missing: id"
assert 'a' in cols, "Missing: a"
assert 'b' in cols, "Missing: b"
assert 'products' in cols, "Missing: products"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
rows = result_5.collect()
assert len(rows) == 1, "Expected 1 row"
assert rows[0]['products'][0] == 10, f"products[0] should be 10, got {rows[0]['products'][0]}"
assert rows[0]['products'][1] == 40, f"products[1] should be 40, got {rows[0]['products'][1]}"
assert rows[0]['products'][2] == 90, f"products[2] should be 90, got {rows[0]['products'][2]}"
print(f"Problem 5 passed  ({len(rows)} rows)")

## Problem 6 - array utility functions

Using `df_transactions`, group by `franchiseID` and use `F.collect_set('product')` to build an array of unique products per franchise.  
Then apply the following in sequence:
- `F.array_distinct()` on the collected set
- `F.array_sort()` to sort alphabetically
- `F.array_size()` to count unique products
- `F.array_join(col, ', ')` to produce a comma-separated string

**Expected columns:** `franchiseID`, `unique_products_sorted`, `product_count`, `products_string`

In [ ]:
result_6 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 6 ---------
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'franchiseid' in cols, "Missing: franchiseID"
assert 'unique_products_sorted' in cols, "Missing: unique_products_sorted"
assert 'product_count' in cols, "Missing: product_count"
assert 'products_string' in cols, "Missing: products_string"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# product_count must be > 0 for all rows
zero_count = result_6.filter(F.col('product_count') <= 0)
assert zero_count.count() == 0, "product_count must be > 0 for all rows"
# products_string must be non-null
null_str = result_6.filter(F.col('products_string').isNull())
assert null_str.count() == 0, "products_string must not be null"
# product_count must match size of unique_products_sorted
mismatch = result_6.filter(F.col('product_count') != F.array_size(F.col('unique_products_sorted')))
assert mismatch.count() == 0, "product_count must match array_size of unique_products_sorted"
print(f"Problem 6 passed  ({cnt} rows)")

## Problem 7 - from_json and schema_of_json

Create the following test DataFrame with a JSON string column:

```python
df_json = spark.createDataFrame(
    [('\'{"name":"Alice","age":30}\',), ('\'{"name":"Bob","age":25}\',)],
    ['json_str']
)
```

Use `F.schema_of_json()` to infer the schema from a sample literal, then use `F.from_json()` to parse the `json_str` column into a struct column called `parsed`.  
Extract nested fields into separate columns `name` and `age`.  
**Expected columns:** `json_str`, `parsed`, `name`, `age`

In [ ]:
result_7 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 7 ---------
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'json_str' in cols, "Missing: json_str"
assert 'parsed' in cols, "Missing: parsed"
assert 'name' in cols, "Missing: name"
assert 'age' in cols, "Missing: age"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
rows = result_7.orderBy('name').collect()
assert len(rows) == 2, "Expected 2 rows"
assert rows[0]['name'] == 'Alice', f"First row name should be Alice, got {rows[0]['name']}"
assert rows[0]['age'] == 30, f"Alice age should be 30, got {rows[0]['age']}"
assert rows[1]['name'] == 'Bob', f"Second row name should be Bob, got {rows[1]['name']}"
print(f"Problem 7 passed  ({len(rows)} rows)")

## Problem 8 - map utility functions

Using `df_transactions`, use `F.create_map(F.col('product'), F.col('totalPrice'))` to build a map column called `product_price_map` for each transaction.  
Then:
- Use `F.map_keys()` to extract the keys into a column called `map_keys`
- Use `F.map_values()` to extract the values into a column called `map_values`
- Use `F.map_from_arrays()` to reconstruct the map from those key and value arrays

**Expected columns:** `transactionID`, `product_price_map`, `map_keys`, `map_values`

In [ ]:
result_8 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 8 ---------
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
assert 'transactionid' in cols, "Missing: transactionID"
assert 'product_price_map' in cols, "Missing: product_price_map"
assert 'map_keys' in cols, "Missing: map_keys"
assert 'map_values' in cols, "Missing: map_values"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_8.count()
assert cnt > 0, "Got 0 rows"
# each transaction row creates a single-entry map, so map_keys should have size 1
wrong_size = result_8.filter(F.size(F.col('map_keys')) != 1)
assert wrong_size.count() == 0, "Each transaction should produce a map_keys array of size 1"
# map_values should also have size 1
wrong_vals = result_8.filter(F.size(F.col('map_values')) != 1)
assert wrong_vals.count() == 0, "Each transaction should produce a map_values array of size 1"
# map_keys should not contain nulls
# F.array_contains(..., None) throws DATATYPE_MISMATCH.NULL_TYPE on Spark
null_keys = result_8.filter(F.exists(F.col('map_keys'), lambda k: k.isNull()))
assert null_keys.count() == 0, "map_keys should not contain null values"
print(f"Problem 8 passed  ({cnt} rows)")